# Questão 1 — EDA inicial da tabela `orders`

**Premissas obrigatórias:** usar apenas `orders.csv`, sem nenhuma limpeza/tratamento — apenas observar, agregar e descrever.

In [1]:
import duckdb
import pandas as pd

con = duckdb.connect()
csv_path = "../database/orders.csv"
orders = f"read_csv_auto('{csv_path}')"

## Parte 1 — Visão geral (linhas, colunas, intervalo de datas)

In [2]:
overview = con.execute(f"""
    SELECT
        COUNT(*) AS total_linhas,
        (SELECT COUNT(*) FROM (DESCRIBE SELECT * FROM {orders})) AS total_colunas,
        MIN(created_at) AS data_minima,
        MAX(created_at) AS data_maxima
    FROM {orders}
""").fetchdf()
overview

,total_linhas,total_colunas,data_minima,data_maxima
0,48998,13,2020-01-01 01:19:28,2026-12-31 23:43:09


## Parte 2 — Estatísticas da coluna `total`

In [3]:
stats_total = con.execute(f"""
    SELECT
        MIN(total) AS total_min,
        MAX(total) AS total_max,
        AVG(total) AS total_medio,
        COUNT(*) - COUNT(total) AS qtd_nulos_total
    FROM {orders}
""").fetchdf()
stats_total

,total_min,total_max,total_medio,qtd_nulos_total
0,32.62,127262.02,28704.992077,0


## Apoio ao diagnóstico — nulos por coluna

In [4]:
df = pd.read_csv(csv_path)
df.isnull().sum()

id                     0
order_number           0
channel                0
customer_id            0
salesperson_id     24131
location_id            0
status                 0
subtotal               0
discount_amount        0
total                  0
placed_at              0
created_at             0
updated_at             0
dtype: int64

## Apoio ao diagnóstico — possíveis outliers em `total` (regra IQR 1.5x)

In [5]:
iqr = con.execute(f"""
    WITH s AS (
        SELECT
            quantile_cont(total, 0.25) AS q1,
            quantile_cont(total, 0.75) AS q3
        FROM {orders}
    )
    SELECT q1, q3, (q3-q1) AS iqr,
           (q1 - 1.5*(q3-q1)) AS limite_inferior,
           (q3 + 1.5*(q3-q1)) AS limite_superior
    FROM s
""").fetchdf()
iqr

,q1,q3,iqr,limite_inferior,limite_superior
0,13171.235,40941.8825,27770.6475,-28484.73625,82597.85375


In [6]:
lim_sup = iqr['limite_superior'][0]
lim_inf = iqr['limite_inferior'][0]
acima = (df['total'] > lim_sup).sum()
abaixo = (df['total'] < lim_inf).sum()
negativos = (df['total'] < 0).sum()
print(f"Pedidos acima do limite superior ({lim_sup:.2f}): {acima}")
print(f"Pedidos abaixo do limite inferior ({lim_inf:.2f}): {abaixo}")
print(f"Pedidos com total negativo: {negativos}")

Pedidos acima do limite superior (82597.85): 452
Pedidos abaixo do limite inferior (-28484.74): 0
Pedidos com total negativo: 0


## Apoio ao diagnóstico — checagem de datas futuras

In [7]:
from datetime import datetime
hoje = datetime(2026, 8, 12)
df['created_at'] = pd.to_datetime(df['created_at'])
futuros = (df['created_at'] > hoje).sum()
print(f"Pedidos com created_at no futuro (após {hoje.date()}): {futuros}")
print(f"Data máxima encontrada: {df['created_at'].max()}")

Pedidos com created_at no futuro (após 2026-08-12): 4305
Data máxima encontrada: 2026-12-31 23:43:09


## Resultados obtidos

- **Total de linhas:** 48.998
- **Total de colunas:** 13
- **Intervalo de created_at:** 2020-01-01 01:19:28 até 2026-12-31 23:43:09
- **total — mínimo:** 32,62 | **máximo:** 127.262,02 | **médio:** 28.704,99

Ver diagnóstico completo (Questão 1.3) em `answers/respostas.md`.